# Nuclear Receptor Ligand Analysis - Family-Based Approach
## Integrated analysis with molecular properties for each NR family

## 1. Fetch UniProt IDs for Nuclear Receptors

In [ ]:
import requests
import pandas as pd

# 48 canonical human nuclear receptor gene symbols
nr_genes = [
    "AR", "ESR1", "ESR2", "NR3C1", "NR3C2", "PGR",
    "PPARA", "PPARG", "PPARD",
    "RXRA", "RXRB", "RXRG",
    "THRA", "THRB",
    "RARA", "RARB", "RARG",
    "VDR", "NR1H4", "NR1H3", "NR1H2",
    "RORA", "RORB", "RORC",
    "NR1D1", "NR1D2",
    "NR4A1", "NR4A2", "NR4A3",
    "HNF4A", "HNF4G",
    "NR2C1", "NR2C2",
    "NR2E1", "NR2E3",
    "NR2F1", "NR2F2", "NR2F6",
    "NR1I2", "NR1I3",
    "NR5A1", "NR5A2",
    "NR6A1",
    "NR0B1", "NR0B2",
    "ESRRA", "ESRRB", "ESRRG"
]

def fetch_uniprot_ids(gene_list):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    records = []

    for gene in gene_list:
        query = f"(gene:{gene}) AND (organism_id:9606) AND (reviewed:true)"
        params = {
            "query": query,
            "format": "json",
            "fields": "accession,id,gene_names,protein_name"
        }
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        data = r.json()

        for entry in data.get("results", []):
            acc = entry.get("primaryAccession")
            pid = entry.get("uniProtkbId")
            gnames = []
            for g in entry.get("genes", []):
                if "geneName" in g:
                    gnames.append(g["geneName"]["value"])
            pname = (
                entry.get("proteinDescription", {})
                .get("recommendedName", {})
                .get("fullName", {})
                .get("value", "")
            )
            records.append({
                "QueryGene": gene,
                "UniProtID": acc,
                "EntryName": pid,
                "ProteinName": pname,
                "GeneNames": ",".join(gnames)
            })
    return pd.DataFrame(records)

# Run
df_nrs = fetch_uniprot_ids(nr_genes)
print(f"✅ Retrieved {len(df_nrs)} UniProt IDs for {df_nrs['QueryGene'].nunique()} NR genes")
print(df_nrs.head())

# Save
df_nrs.to_csv("human_nuclear_receptors_uniprot.csv", index=False)

## 2. Load Your Ligand Database with NR Family Classification

In [ ]:
import os

# === LOAD YOUR DATA ===
# The file is already uploaded and ready to use!
input_file = "/mnt/user-data/uploads/ligands_with_final_chemical_classification.csv"

# Check if file exists
if not os.path.exists(input_file):
    print("❌ ERROR: Data file not found!")
    print(f"   Looking for: {input_file}")
    raise FileNotFoundError(f"Cannot find: {input_file}")

df = pd.read_csv(input_file)

print(f"✅ Total ligands loaded: {len(df)}")
print(f"\nColumns in dataset: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

## 3. Check NR Family Distribution

In [ ]:
# Check if NR_Family column exists
if "NR_Family" not in df.columns:
    raise KeyError("⚠️ 'NR_Family' column missing in the CSV file.")

# See distribution of families
family_counts = df["NR_Family"].value_counts()
print("📊 NR Family Distribution:")
print(family_counts)
print(f"\n✅ Total families: {df['NR_Family'].nunique()}")

## 4. Define Molecular Property Calculator

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

def calculate_molecular_properties(smiles):
    """Calculate comprehensive molecular properties from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            "MolWt": None,
            "HeavyAtomCount": None,
            "RingCount": None,
            "LogP": None,
            "TPSA": None,
            "HBD": None,
            "HBA": None,
            "RotatableBonds": None,
            "StereoCenters": None,
            "AromaticRings": None,
            "Lipinski_Pass": None,
            "Veber_Pass": None,
            "QED": None,
            "NumHeteroatoms": None,
            "FractionCSP3": None,
            "MolMR": None
        }

    # Basic descriptors
    mol_wt = Descriptors.MolWt(mol)
    heavy_atoms = Descriptors.HeavyAtomCount(mol)
    ring_count = Descriptors.RingCount(mol)

    # Physicochemical properties
    logp = Crippen.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    # Structural complexity
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    stereo_centers = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    
    # Additional descriptors
    num_heteroatoms = Lipinski.NumHeteroatoms(mol)
    fraction_csp3 = Lipinski.FractionCSP3(mol)
    mol_mr = Crippen.MolMR(mol)

    # Drug-likeness rules
    lipinski_pass = (mol_wt <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    veber_pass = (rot_bonds <= 10 and tpsa <= 140)
    
    # QED (Quantitative Estimate of Drug-likeness)
    try:
        from rdkit.Chem import QED
        qed = QED.qed(mol)
    except:
        qed = None

    return {
        "MolWt": mol_wt,
        "HeavyAtomCount": heavy_atoms,
        "RingCount": ring_count,
        "LogP": logp,
        "TPSA": tpsa,
        "HBD": hbd,
        "HBA": hba,
        "RotatableBonds": rot_bonds,
        "StereoCenters": stereo_centers,
        "AromaticRings": aromatic_rings,
        "Lipinski_Pass": lipinski_pass,
        "Veber_Pass": veber_pass,
        "QED": qed,
        "NumHeteroatoms": num_heteroatoms,
        "FractionCSP3": fraction_csp3,
        "MolMR": mol_mr
    }

print("✅ Molecular property calculator defined")

## 5. Calculate Properties for ALL Ligands (Master Dataset)

In [ ]:
print("⏳ Calculating molecular properties for all ligands...")

descriptor_data = []
for idx, row in df.iterrows():
    smiles = row.get("SMILES", "")
    props = calculate_molecular_properties(smiles)
    descriptor_data.append(props)
    
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{len(df)} ligands...")

# Add descriptors as new columns
desc_df = pd.DataFrame(descriptor_data)
df_full = pd.concat([df.reset_index(drop=True), desc_df.reset_index(drop=True)], axis=1)

# Save master file with all properties
output_master = "all_ligands_with_molecular_properties.csv"
df_full.to_csv(output_master, index=False)

print(f"\n✅ Master file saved: {output_master}")
print(f"✅ Total ligands processed: {len(df_full)}")
print(f"✅ Total columns: {len(df_full.columns)}")

## 6. Create Separate Files for Each NR Family

In [ ]:
import os

# Create directory for family-specific files
output_dir = "family_specific_analysis"
os.makedirs(output_dir, exist_ok=True)

# Get unique families
families = df_full["NR_Family"].dropna().unique()

print(f"📁 Creating separate files for {len(families)} families...\n")

family_summary = []

for family in families:
    # Filter data for this family
    family_df = df_full[df_full["NR_Family"] == family].copy()
    
    # Save to separate file
    family_clean = str(family).lower().replace(" ", "_").replace("/", "_")
    output_file = os.path.join(output_dir, f"{family_clean}_molecular_properties.csv")
    family_df.to_csv(output_file, index=False)
    
    # Calculate summary statistics
    summary = {
        "Family": family,
        "N_Ligands": len(family_df),
        "Mean_MolWt": family_df["MolWt"].mean(),
        "Mean_LogP": family_df["LogP"].mean(),
        "Mean_TPSA": family_df["TPSA"].mean(),
        "Mean_HBD": family_df["HBD"].mean(),
        "Mean_HBA": family_df["HBA"].mean(),
        "Mean_QED": family_df["QED"].mean(),
        "Lipinski_Pass_Rate": family_df["Lipinski_Pass"].mean() * 100,
        "Veber_Pass_Rate": family_df["Veber_Pass"].mean() * 100,
        "File": output_file
    }
    family_summary.append(summary)
    
    print(f"✅ {family}: {len(family_df)} ligands → {output_file}")

# Create summary table
summary_df = pd.DataFrame(family_summary)
summary_df = summary_df.sort_values("N_Ligands", ascending=False)
summary_file = "family_summary_statistics.csv"
summary_df.to_csv(summary_file, index=False)

print(f"\n📊 Summary statistics saved to: {summary_file}")
print("\n" + "="*80)
print(summary_df.to_string(index=False))

## 7. Comparative Analysis Across Families

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Molecular Property Comparison Across NR Families', fontsize=16, fontweight='bold')

# Properties to compare
properties = ['MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'QED']
titles = ['Molecular Weight', 'LogP (Lipophilicity)', 'TPSA', 
          'H-Bond Donors', 'H-Bond Acceptors', 'QED (Drug-likeness)']

for idx, (prop, title) in enumerate(zip(properties, titles)):
    ax = axes[idx // 3, idx % 3]
    
    # Box plot for each family
    df_full.boxplot(column=prop, by='NR_Family', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('NR Family')
    ax.set_ylabel(prop)
    plt.sca(ax)
    plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('family_comparison_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Comparative analysis plot saved: family_comparison_boxplots.png")

## 8. Drug-likeness Analysis by Family

In [ ]:
# Calculate drug-likeness pass rates by family
druglikeness = df_full.groupby('NR_Family').agg({
    'Lipinski_Pass': ['sum', 'count', 'mean'],
    'Veber_Pass': ['sum', 'mean'],
    'QED': 'mean'
}).round(3)

druglikeness.columns = ['_'.join(col).strip() for col in druglikeness.columns.values]
druglikeness = druglikeness.sort_values('Lipinski_Pass_mean', ascending=False)

print("\n📊 Drug-likeness Pass Rates by Family:")
print(druglikeness)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Lipinski pass rate
families_sorted = druglikeness.index
lipinski_rates = druglikeness['Lipinski_Pass_mean'] * 100
ax1.barh(range(len(families_sorted)), lipinski_rates, color='steelblue')
ax1.set_yticks(range(len(families_sorted)))
ax1.set_yticklabels(families_sorted)
ax1.set_xlabel('Lipinski Pass Rate (%)')
ax1.set_title('Lipinski Rule of 5 Pass Rate by Family')
ax1.axvline(x=50, color='red', linestyle='--', alpha=0.5)

# Veber pass rate
veber_rates = druglikeness['Veber_Pass_mean'] * 100
ax2.barh(range(len(families_sorted)), veber_rates, color='seagreen')
ax2.set_yticks(range(len(families_sorted)))
ax2.set_yticklabels(families_sorted)
ax2.set_xlabel('Veber Rule Pass Rate (%)')
ax2.set_title('Veber Rule Pass Rate by Family')
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('druglikeness_by_family.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Drug-likeness plot saved: druglikeness_by_family.png")

## 9. Scaffold Analysis for Each Family

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

def get_scaffold(smiles):
    """Extract Murcko scaffold from SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            scaffold = MurckoScaffold.GetScaffoldForMol(mol)
            return Chem.MolToSmiles(scaffold)
        else:
            return None
    except:
        return None

# Calculate scaffolds for all ligands
print("⏳ Extracting Murcko scaffolds...")
df_full["Scaffold"] = df_full["SMILES"].apply(get_scaffold)

# Analyze scaffolds by family
scaffold_summary = []

for family in df_full["NR_Family"].dropna().unique():
    family_df = df_full[df_full["NR_Family"] == family]
    scaffold_counts = family_df["Scaffold"].value_counts()
    
    summary = {
        "Family": family,
        "Total_Ligands": len(family_df),
        "Unique_Scaffolds": scaffold_counts.nunique(),
        "Most_Common_Scaffold": scaffold_counts.index[0] if len(scaffold_counts) > 0 else None,
        "Most_Common_Count": scaffold_counts.iloc[0] if len(scaffold_counts) > 0 else 0,
        "Scaffold_Diversity": scaffold_counts.nunique() / len(family_df) if len(family_df) > 0 else 0
    }
    scaffold_summary.append(summary)

scaffold_df = pd.DataFrame(scaffold_summary)
scaffold_df = scaffold_df.sort_values("Scaffold_Diversity", ascending=False)

print("\n📊 Scaffold Diversity by Family:")
print(scaffold_df.to_string(index=False))

scaffold_df.to_csv("scaffold_diversity_by_family.csv", index=False)
print("\n✅ Scaffold analysis saved: scaffold_diversity_by_family.csv")

## 10. Generate Fingerprints for Similarity Analysis (Optional)

In [ ]:
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# Initialize Morgan fingerprint generator
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

def get_fingerprint(smiles):
    """Generate Morgan fingerprint from SMILES"""
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp = morgan_gen.GetFingerprint(mol)
        return list(fp.ToList())
    else:
        return None

print("⏳ Generating molecular fingerprints...")
df_full["Fingerprint"] = df_full["SMILES"].apply(get_fingerprint)

# Filter valid molecules
valid_df = df_full[df_full["Fingerprint"].notnull()].reset_index(drop=True)
print(f"✅ Valid ligands with fingerprints: {len(valid_df)} / {len(df_full)}")

# Create fingerprint DataFrame
fp_df = pd.DataFrame(valid_df["Fingerprint"].to_list(), columns=[f"Bit_{i}" for i in range(2048)])

# Combine and save
final_fp_df = pd.concat([valid_df.drop(columns=["Fingerprint"]), fp_df], axis=1)
output_fp = "all_ligands_with_fingerprints.csv"
final_fp_df.to_csv(output_fp, index=False)

print(f"💾 Fingerprint dataset saved to: {output_fp}")

## 11. Summary Report

In [ ]:
print("\n" + "="*80)
print("📊 ANALYSIS COMPLETE - SUMMARY REPORT")
print("="*80)

print(f"\n1️⃣ Master Dataset:")
print(f"   - Total ligands: {len(df_full)}")
print(f"   - Total NR families: {df_full['NR_Family'].nunique()}")
print(f"   - File: all_ligands_with_molecular_properties.csv")

print(f"\n2️⃣ Family-Specific Files:")
print(f"   - Created {len(families)} separate files in '{output_dir}/' directory")
print(f"   - Summary statistics: family_summary_statistics.csv")

print(f"\n3️⃣ Analysis Files Generated:")
print(f"   - family_comparison_boxplots.png")
print(f"   - druglikeness_by_family.png")
print(f"   - scaffold_diversity_by_family.csv")
print(f"   - all_ligands_with_fingerprints.csv")

print(f"\n4️⃣ Top 5 Families by Ligand Count:")
top_families = df_full['NR_Family'].value_counts().head()
for family, count in top_families.items():
    print(f"   - {family}: {count} ligands")

print("\n" + "="*80)
print("✅ All analyses completed successfully!")
print("="*80)